In [ ]:
# import torch
# import numpy as np
# import random
# from datasets import load_dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForSeq2SeqLM,
#     Seq2SeqTrainer,
#     Seq2SeqTrainingArguments,
#     DataCollatorForSeq2Seq,
# )
# from peft import LoraConfig, get_peft_model, TaskType
# !pip install -U torchao -q

In [ ]:
# --- Config ---
MODEL_NAME = "google/flan-t5-large"   # change per run: small / base / large
DATASET_NAME = "tatsu-lab/alpaca"
SUBSET_SIZE = 1000
SEED = 42
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128
OUTPUT_DIR = "./output_alpaca_1k"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# --- Load and prepare data ---
print("Loading dataset...")
dataset = load_dataset(DATASET_NAME, split="train")
dataset = dataset.shuffle(seed=SEED).select(range(SUBSET_SIZE))

def format_example(example):
    # Alpaca fields: instruction, input, output (Dolly uses instruction/context/response)
    if example.get("input"):
        prompt = f"Instruction: {example['instruction']}\nContext: {example['input']}"
    else:
        prompt = f"Instruction: {example['instruction']}"
    return {"input_text": prompt, "target_text": example["output"]}

dataset = dataset.map(format_example)

split = dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

Loading dataset...
Train size: 900, Eval size: 100


In [ ]:
# --- Tokenizer and preprocessing ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=example["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(preprocess, remove_columns=eval_dataset.column_names)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# !pip install -U torchao -q

In [ ]:
# --- Load model + apply LoRA (re-run this cell fresh before EVERY run) ---
print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading model...


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 2,359,296 || all params: 785,509,376 || trainable%: 0.3004


In [ ]:
# --- Training setup: LOCKED FINAL CONFIG ---
PER_DEVICE_TRAIN_BS = 3
GRAD_ACCUM_STEPS = 2
PER_DEVICE_EVAL_BS = 2

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BS,
    max_steps=300,          # fixed step count, same across all models
    learning_rate=3e-3,     # matches the actual locked config used in the validated Dolly runs
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to="none",
    predict_with_generate=True,
    fp16=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
# --- Train ---
print("Starting training...")
trainer.train()

# --- Evaluate ---
print("Evaluating...")
eval_results = trainer.evaluate()
print(eval_results)
print(f"\nFinal eval loss: {eval_results.get('eval_loss')}")

Starting training...


Epoch,Training Loss,Validation Loss
1,2.055508,0.902090
2,1.713885,0.845280


Evaluating...


Training Loss,Validation Loss,Epoch
1.713885,0.845280,2


{'eval_loss': 0.84527987241745}

Final eval loss: 0.84527987241745
